# Optimizing Marker Expression for Late MgkPro.

## Prepare Notebook.

### Autoreload Libraries.

In [ ]:
%reload_ext autoreload
%autoreload 2

### Import the Libraries.

In [ ]:
from functools import partial
import sys

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import torch


import hydra
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
import torch

import sc_exp_design

sys.path.insert(0, "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/collab-goettgens-SFC/shared_utils")

from experiment_utils import get_forward_model, generate_with_condition, generated_density_knn, get_adata_from_idx
from data_utils import (
    standardize_array_and_write_to_adata, get_condition_data_from_file,
    get_protocol_tranformations, get_target_adata_marker_opt
)
from inverse_utils import (
    map_df, l2_loss, l1_loss, hinge_loss, cauchy_loss, loss_fn_factory_marker_opt,
    linear_scheduler_with_warmup, constraint_fn_factory
)
from lambda_schedulers import ReciprocalDecayScheduler
from plot_utils import plot_all_markers_overlay


## Load Stuff.

### Load Configurations.

In [ ]:
with hydra.initialize(
    config_path="../../config"
):
    config = hydra.compose(
        config_name="run_inverse_constrained",
        overrides=[
            "paths=bloodplus_fm",
            "annotation=bloodplus",
            "sampling.N=10",
            "sampling.num_time_steps=20",
            "forward_model.num_time_steps=20",
            "constraints=default_all_cols"
        ]
    )

### Load Prior Model.

In [ ]:
# initialize prior models
prior_cfm = sc_exp_design.models.FlowMatchingWithScore.load(
    config.paths.prior_flow_path
)
prior_fm = sc_exp_design.models.FlowMap.load(
    config.paths.prior_flow_map_path
)
prior_cfm.velocity_field.eval()
prior_fm.flow_map.eval()

# show modules
print(prior_cfm.velocity_field)
print(prior_fm.flow_map)

### Load Forward Model.

In [ ]:
forward_model, (
    phi_model,
    target_prediction_model
) = get_forward_model(config)
forward_model.target_prediction_model.resc_model["model"].eval()
forward_model.forward_model.velocity_field.eval()
print(forward_model.target_prediction_model.resc_model["model"])
print(forward_model.forward_model.velocity_field)

### Load Reference Media Concentrations.

In [ ]:
condition_metadata_path = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/data/gdrive/table_metadata_theis_unmix8.xlsx" 
protocol_data, condition_adata = get_condition_data_from_file(
    phi_model.data_manager,
    condition_metadata_path,
    config.annotation.protocol_columns,
    config.annotation.log1p_exp_cols,
    config.annotation.log21p_exp_cols,
    config.annotation.protocol_obs_key_added,
    config.annotation.protocol_sep,
    config.annotation.one_hot_uns_key_added,
    config.annotation.protocol_obsm_key,
)
condition_adata

### Read Original Annotation File.

In [ ]:
experiment_id = "LPHO012"
additional_df_cols = ["experiment_number"]

condition_df = pd.read_excel(condition_metadata_path)
condition_df.columns = [col.replace("/", "_") for col in condition_df.columns]

all_df_cols = additional_df_cols + config.annotation.protocol_columns
lph_df = condition_df.loc[condition_df.experiment_id == experiment_id, all_df_cols]
lph_df

## Prepare Data.

### Concatenate Full Data.

In [ ]:
# data for cellular response prediction model (full)
train_adata_phi = phi_model.train_data.adata
val_adata_phi = next(iter(phi_model.validation_data.values())).adata
adata_phi = sc.concat((train_adata_phi, val_adata_phi), uns_merge="same")
print(f"Full data shape: {adata_phi.shape}")

### Concatenate Annotated Subset.

In [ ]:
# data for classifier model (subset)
train_adata_g = target_prediction_model.train_data.adata
val_adata_g = target_prediction_model.validation_data.adata
adata_g = sc.concat((train_adata_g, val_adata_g), uns_merge="same")
print(f"Subset data shape: {adata_g.shape}")

### Fit Label Encoder.

In [ ]:
ct_le = LabelEncoder()
ct_values = target_prediction_model.train_data.adata.obs["cell_type"].values
ct_le.fit(ct_values)
classes = ct_le.classes_.tolist()

## Get Target Values.

### Prepare Target Adatas.

In [ ]:
# define target markers
target_marker_names = [
    # "CD34",
    "CD41a",
    # "CD56",
    # "EPCR"
]
target_morph_feat_names = [
    # "SSC-A :: SSC - Area",
    # "FSC-A :: FSC - Area",
]

# define aggregation functions
agg_fn_dict = {
    "mean": np.mean,
    "median": np.median,
    "min": np.min,
    "max": np.max,
    "pop": lambda x: x,
}

# define filter configurations
filter_dict = {
    "experiment_number": 210,
    "cell_type": "late_MgkPro",
}
target_quantile = 75

target_adatas_dict = {}
for agg_id, agg_fn in agg_fn_dict.items():
    # prepare keyword arguments for aggregation
    agg_fn_kwargs = {"axis": 0, "keepdims": True} if agg_id != "pop" else {}

    target_adata, query_adata = get_target_adata_marker_opt(
        train_adata_phi,
        adata_g,
        agg_fn,
        target_marker_names,
        target_morph_feat_names,
        config.annotation.scatter_columns,
        filter_dict,
        target_quantile,
        agg_fn_kwargs=agg_fn_kwargs,
    )
    target_adatas_dict[agg_id] = target_adata
target_feats_mask = query_adata.uns["target_feats_mask"]

### Plot Retained Cells.

In [ ]:
ref_main_idx = adata_g.obs.index.map(lambda e: e in query_adata.obs.index)
adata_g.obs["is_ref_pop"] = ref_main_idx
target_feat_names = target_marker_names + target_morph_feat_names
sc.pl.embedding(adata_g, "umap", color=["is_ref_pop"] + target_feat_names, s=20)

### Visualizing Resulting Targets After Transformation.

In [ ]:
cell_state_rep_orig = "X_channel+X_scatter"
cell_state_rep_std = "X_channel_standardized+X_scatter_standardized"
all_feat_names = adata_g.var_names.to_list() + config.annotation.scatter_columns
for target_type, target_adata in target_adatas_dict.items():
    print(f"="*80)
    print(f"Target Type {target_type}")
    for idx, var_name in enumerate(all_feat_names):
        if var_name in target_feat_names:
            val_orig = target_adata.obsm[cell_state_rep_orig][:, idx].mean(0)
            val_std = target_adata.obsm[cell_state_rep_std][:, idx].mean(0)
            print(f"{var_name} -> Before {val_orig.item()}, Rescaled: {val_std.item()}")

### Prepare Tensor Representations for Target States.

In [ ]:
cell_state_rep = "X_channel_standardized+X_scatter_standardized"
device = "cuda"
dtype = torch.float32
xopt_dict_torch = {
    target_type: torch.from_numpy(target_adata.obsm[cell_state_rep]).\
        to(dtype).to(device)
    for target_type, target_adata in target_adatas_dict.items()
}
for target_type, xopt_torch in xopt_dict_torch.items():
    print(f"{target_type} -> {xopt_torch.shape}, {xopt_torch[:, target_feats_mask]}")


## Run Optimization (Loss Guidance).

### Prepare Losses.

In [ ]:
loss_registry = {
    "l2": l2_loss,
    "l1": l1_loss,
    "hinge": hinge_loss,
    "cauchy": cauchy_loss,
}
loss_fn_dict = {}
non_linearity = torch.nn.Identity()
for target_type in ["max", "min", "mean", "median"]:
    for loss_type, loss_fn in loss_registry.items():
        for agg_type in ["pop", "cell"]:
            xstar = xopt_dict_torch[target_type]          # (1, D)
            loss_fn_comp, noise = loss_fn_factory_marker_opt(
                config, loss_fn, xstar, non_linearity, phi_model,
                target_feats_mask, agg_type=agg_type
            )
            loss_fn_dict[f"{target_type}_{loss_type}_{agg_type}"] = {"loss_fn": loss_fn_comp, "noise": noise}
print(loss_fn_dict.keys())

### Prepare Penalization.

In [ ]:
# compile constraints
config.constraints.c_scheduler_kwargs.vmax = 5.0
print(config.constraints.lbound_dict)
print(config.constraints.ubound_dict)
print(config.constraints.c_scheduler_kwargs)
print(config.constraints.use_exponential_penalty)
compute_constraints = constraint_fn_factory(config, forward_model.forward_model.device, get_protocol_tranformations)
c_scheduler = partial(
    linear_scheduler_with_warmup,
    t_warmup=config.constraints.c_scheduler_kwargs.t_warmup,
    vmin=config.constraints.c_scheduler_kwargs.vmin,
    vmax=config.constraints.c_scheduler_kwargs.vmax,
)

### Sample from Inverse Model.

In [ ]:
# initialize loss guidance
guided_flow = sc_exp_design.inverse.LossGuidedFlow(prior_cfm, prior_flow_map=prior_fm)

# prepare scheduler
lmax = 1.0
scheduler_instance = ReciprocalDecayScheduler(lmax=lmax)
scheduler = lambda t: scheduler_instance.compute_lambda_t(t)

# get loss function
agg = "min"
loss_type = "pop"
loss_name = f"{agg}_l2_{loss_type}"
adata_ref = target_adatas_dict[agg]
loss_fn = loss_fn_dict[loss_name]["loss_fn"]

# prepare sampling configurations
num_time_steps=20
solver_kwargs = {"method": "euler", "atol": 5e-5, "rtol": 5e-5}

# sample from inverse problem
use_penalization = True
traj, loss_history, lambda_history = guided_flow.sample_posterior(
    config.sampling.N,
    loss_fn,
    reg_fn_lists=compute_constraints if use_penalization else None,
    c_scheduler=c_scheduler if use_penalization else None,
    lambda_scheduler=scheduler,
    num_time_steps=num_time_steps,
    solver_kwargs=solver_kwargs
)
print(f"{traj.shape=}, {loss_history.shape=}, {lambda_history.shape=}")

### Visualize loss history.

In [ ]:
# loss_history is a list of length (num_time_steps+1), each element a numpy array (N,)
loss_arr = np.array(loss_history.cpu())   # shape (T+1, N)
T = loss_arr.shape[0] - 1
steps = np.arange(T+1)

# Plot individual samples (first up to 10)
plt.figure(figsize=(5, 3))
N_samples = loss_arr.shape[1]
for i in range(min(N_samples, 10)):
    plt.plot(steps, loss_arr[:, i], alpha=0.6, linewidth=1, label=f'Sample {i+1}')

# Add mean ± std
mean_loss = loss_arr.mean(axis=1)
std_loss = loss_arr.std(axis=1)
plt.plot(steps, mean_loss, 'k-', linewidth=2, label='Mean')
plt.fill_between(steps, mean_loss - std_loss, mean_loss + std_loss, 
                    alpha=0.2, color='gray', label='±1 std')

plt.xlabel('ODE step')
plt.ylabel('Loss (final condition)')
plt.title(f'Loss evolution: {loss_name}')
# plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Visualize Generated Samples.

In [ ]:
# Clip negative values to 0 (optional)
samples = traj[-1]
samples = np.maximum(samples, 0.0)

# Create clustermap with original values
g = sns.clustermap(
    samples,
    xticklabels=config.annotation.protocol_columns,
    figsize=(6, 6),
    cbar_pos=(0.02, 0.8, 0.03, 0.15),
    dendrogram_ratio=(0.1, 0.2),
    vmin=0.0,
    vmax=10.0,
    
)
g.fig.suptitle(f"Loss Guidance – {target_type}", y=1.02)
# Add title
plt.show()


### Tranform Samples Back to their Original Scale.

In [ ]:
# get transformations
protocol_transf = get_protocol_tranformations(
    config.annotation.protocol_columns,
    log1p_exp_cols=config.annotation.log1p_exp_cols,
    log21p_exp_cols=config.annotation.log21p_exp_cols,
    inverse=True
)


samples = np.maximum(samples, 0)
raw_opt_df = pd.DataFrame(samples, columns=config.annotation.protocol_columns)
opt_df = map_df(raw_opt_df, protocol_transf)
print("="*80)
print(config.constraints.ubound_dict)
opt_df

### Sakurai Medium.

In [ ]:
lph_df[lph_df.experiment_number == 210][config.annotation.protocol_columns]

### Query Forward Model.

In [ ]:
# settings for generation
plot_umap_density = True
num_time_steps = 100
solver_kwargs = {"method": "euler", "atol": 5e-5, "rtol": 5e-5}
min_loss_idx = 5
density_type = "channel" # "scatter", "channel"

# Loop over all target types and EGF settings
print("="*80)
print(f"Target: {target_type}")
# query forward model with condition
opt_samples = np.maximum(samples, 0)
fwd_out = generate_with_condition(
    opt_samples,
    forward_model,
    num_time_steps,
    solver_kwargs,
    ct_le,
    num_samples=None,
    noise=noise,
    logger=None,
    n_scatter_feats=6,
)

# prepare cell type proportions
mean_ct_probs = fwd_out["ct_probs"].mean(0)
cell_types = ct_le.classes_

# configure density type
def get_density_input(
    adata_g,
    fwd_out,
    density_type,
):
    if density_type == "joint":
        X_true = adata_g.obsm["X_channel_standardized+X_scatter_standardized"]
        X_generated = fwd_out["X"]
    elif density_type == "scatter":
        X_true = adata_g.obsm["X_scatter_standardized"]
        X_generated = fwd_out["X_scatter"]
    elif density_type == "channel":
        X_true = adata_g.obsm["X_channel_standardized"]
        X_generated = fwd_out["X_channel"]
    else:
        raise ValueError(f"Unsupported density type {density_type}")
    return X_true, X_generated

X_true, X_generated = get_density_input(
    adata_g,
    fwd_out,
    density_type,
)
print(f"{X_true.shape=}, {X_generated.shape=}")

# Plot each generated sample
if plot_umap_density:
    for idx in range(fwd_out["X"].shape[0]):
        print(f"Sample index {idx}")
        # Access the transformed condition values (already in original scale)
        print(opt_df.iloc[idx])

        ct_probs_i = mean_ct_probs[idx]
        fig_bar, ax_bar = plt.subplots(figsize=(8, 4))
        bars = ax_bar.bar(cell_types, ct_probs_i, color='skyblue', edgecolor='black')
        for bar, val in zip(bars, ct_probs_i):
            ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                        f'{val:.3f}', ha='center', va='bottom', fontsize=9)
        ax_bar.set_xlabel('Cell Type')
        ax_bar.set_ylabel('Probability')
        ax_bar.set_title(f'Sample {idx} - Cell Type Probabilities')
        ax_bar.set_ylim(0, max(ct_probs_i) * 1.2)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

        density = generated_density_knn(
            X_true,
            X_generated[idx],
            k=20,
            bandwidth=None
        )
        adata_g.obs["density"] = density
        sc.pl.embedding(
            adata_g, "umap",
            color=["density", "cell_type"] + target_feat_names,
            s=20, legend_loc="on data"
        )


### Visualize.

In [ ]:
# ========== Setup ==========
save_plots = False
plot_dir = "./optimization_plots"
if save_plots:
    os.makedirs(plot_dir, exist_ok=True)

# Indices of target markers (these variables must exist: all_feat_names, target_feat_names)
marker_indices = [all_feat_names.index(m) for m in target_feat_names]

# Reference population matrix (for population targets)
ref_markers = target_adata[:, target_feat_names].X
if hasattr(ref_markers, 'toarray'):
    ref_markers = ref_markers.toarray()

# Simulation parameters (adjust if needed)
num_time_steps = 100
solver_kwargs = {"method": "euler", "atol": 5e-5, "rtol": 5e-5}

# ========== For a single target_type ==========
print("="*80)
print(f"Target: {target_type}")

# Ensure opt_samples is non‑negative (shape: N x D_protocol)
opt_samples = traj[-1]
opt_samples = np.maximum(opt_samples, 0)

# Collect marker expressions from forward simulation
# fwd_out must be defined (e.g., output of generate_with_condition)
gen_markers_list = []
for idx in range(fwd_out["X"].shape[0]):
    X_gen = fwd_out["X"][idx]                     # (n_cells, n_features)
    gen_markers = X_gen[:, marker_indices]        # (n_cells, n_markers)
    gen_markers_list.append(gen_markers)

# Reconstruct target_info
if target_type in ['max', 'mean', 'median']:
    # Point target: extract from xopt_dict_torch
    target_full = xopt_dict_torch[target_type]
    if torch.is_tensor(target_full):
        target_full = target_full.cpu().numpy()
    if target_full.ndim == 1:
        target_full = target_full.reshape(1, -1)
    target_values = target_full[0, marker_indices]
    target_info = (target_values, None)
else:
    # Population target: use reference matrix
    target_info = (None, ref_markers)

# Plot everything
plot_all_markers_overlay(
    gen_markers_list,
    target_info,
    target_type,
    target_feat_names,
    condition_ids=[f'opt_{i}' for i in range(len(gen_markers_list))],
    loss_name=f"Optimization_{target_type}",
    save_dir=plot_dir if save_plots else None
)


### Visualize Condition Space.

In [ ]:
# ----------------------------------------------------------------------
# Prepare unique real condition vectors (one per experiment)
# ----------------------------------------------------------------------
# Real condition vectors (assumed shape (n_cells, n_protocol))
# real_matrix = None
if real_matrix is None:
    real_conds = adata_phi.obsm["condition_concat"]   # or use obs[protocol_columns].values
    exp_nums = adata_phi.obs["experiment_number"].values

    # Get unique condition per experiment
    unique_exps = np.unique(exp_nums)
    real_matrix = []
    exp_labels = []
    for exp in unique_exps:
        mask = exp_nums == exp
        cond = real_conds[mask][0]   # all cells in same exp share the same condition
        real_matrix.append(cond)
        exp_labels.append(exp)
    real_matrix = np.array(real_matrix)   # (n_exp, n_protocol)
    exp_labels = np.array(exp_labels)


# ----------------------------------------------------------------------
# Plot for each optimization run (no density)
# ----------------------------------------------------------------------
# opt_samples shape: (N, n_protocol) – N optimized solutions
opt_samples = np.maximum(opt_samples, 0)
N_opt = opt_samples.shape[0]

# Combine real + this run's generated
combined = np.vstack([real_matrix, opt_samples])

# PCA
pca = PCA(n_components=2)
combined_pca = pca.fit_transform(combined)

n_real = real_matrix.shape[0]
real_pca = combined_pca[:n_real]
generated_pca = combined_pca[n_real:]

# Plot
plt.figure(figsize=(8, 6))

# Real experiments: highlight experiment 210
is_exp210 = (exp_labels == 210)
plt.scatter(real_pca[~is_exp210, 0], real_pca[~is_exp210, 1],
            c='lightgray', label='Other experiments', s=50, alpha=0.7)
plt.scatter(real_pca[is_exp210, 0], real_pca[is_exp210, 1],
            c='red', label='Exp 210', s=100, edgecolors='black', linewidth=1.5, zorder=5)

# Generated conditions (uniform colour, no density)
plt.scatter(generated_pca[:, 0], generated_pca[:, 1],
            c='dodgerblue', s=80, label='Generated solutions',
            edgecolors='black', linewidth=0.5, alpha=0.8)

# Labels and title
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title(f'Condition space: Real vs Generated\nTarget = {target_type}')
plt.legend()
plt.tight_layout()
plt.show()